<div style="background-color:#005500; padding:12px; border-radius:6px;">
    <span style="color:white; font-weight:bold; font-size:16px;">
        This is the discarded Purchase Prediction model - provided as reference.
    </span>
</div>

<div style="background-color:green; padding:12px; border-radius:6px;">
    <span style="color:white; font-size:14px;">
In this notebook:  Initial models achieved near-perfect test performance, which prompted additional diagnostic checks. Given the real-world nature of session-level behavioral data, such performance strongly suggests the presence of a residual proxy or temporal leakage feature. Follow-up analysis focuses on isolating and correcting these signals to ensure true forward-looking prediction
</div>

In [1]:
# Load packages 
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay
)

import matplotlib.pyplot as plt

### 1. LOAD DATA

In [3]:
df = pd.read_csv("data/sessions_df_EDA.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (120000, 19)
Columns: ['session_id', 'customer_id', 'start_time', 'device', 'source', 'country', 'session_duration_min', 'n_events', 'n_distinct_products', 'total_quantity', 'total_revenue_session', 'avg_discount_session', 'add_to_cart', 'checkout', 'page_view', 'target_purchase', 'age', 'signup_date', 'marketing_opt_in']


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   session_id             120000 non-null  int64  
 1   customer_id            120000 non-null  int64  
 2   start_time             120000 non-null  object 
 3   device                 120000 non-null  object 
 4   source                 120000 non-null  object 
 5   country                120000 non-null  object 
 6   session_duration_min   120000 non-null  float64
 7   n_events               120000 non-null  int64  
 8   n_distinct_products    120000 non-null  int64  
 9   total_quantity         120000 non-null  float64
 10  total_revenue_session  120000 non-null  float64
 11  avg_discount_session   120000 non-null  float64
 12  add_to_cart            120000 non-null  int64  
 13  checkout               120000 non-null  int64  
 14  page_view              120000 non-nu

### 2. DEFINE TARGET + DROP LEAKAGE COLUMNS

In [5]:
TARGET_COL = "target_purchase"

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Available columns: {df.columns.tolist()}")

# Leakage columns confirmed in EDA
LEAKAGE_COLS = [
    "checkout",
    "total_revenue_session",
    "total_quantity",
    "avg_discount_session"
]

# Drop leakage columns that exist
leakage_to_drop = [c for c in LEAKAGE_COLS if c in df.columns]
if leakage_to_drop:
    df = df.drop(columns=leakage_to_drop)
    print("Dropped leakage columns:", leakage_to_drop)
else:
    print("No listed leakage columns found to drop.")

# Drop any "duplicate purchase column" if present:
# Heuristic: any other column (besides target) that is perfectly identical to target gets dropped.
dup_target_cols = []
y_tmp = df[TARGET_COL]
for c in df.columns:
    if c == TARGET_COL:
        continue
    # Only consider binary-ish columns
    if df[c].dropna().nunique() <= 2:
        try:
            same = (df[c].fillna(-999) == y_tmp.fillna(-999)).all()
            if same:
                dup_target_cols.append(c)
        except Exception:
            pass

if dup_target_cols:
    df = df.drop(columns=dup_target_cols)
    print("Dropped duplicate target columns:", dup_target_cols)
else:
    print("No duplicate target columns detected.")


Dropped leakage columns: ['checkout', 'total_revenue_session', 'total_quantity', 'avg_discount_session']
No duplicate target columns detected.


### 3. BASIC SANITY CHECKS

In [6]:
print("\nTarget distribution:")
print(df[TARGET_COL].value_counts(dropna=False))
print("\nTarget rate (purchase=1):", df[TARGET_COL].mean())


Target distribution:
target_purchase
0    86420
1    33580
Name: count, dtype: int64

Target rate (purchase=1): 0.2798333333333333


### 4. TRAIN/TEST SPLIT (BEFORE ANY FITTING!)

In [7]:
# 4) TRAIN/TEST SPLIT 

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

print("\nTrain shape:", X_train.shape, " Test shape:", X_test.shape)


Train shape: (90000, 14)  Test shape: (30000, 14)


### 5. DROP "ID-LIKE" COLUMNS FROM MODELING FEATURES

In [8]:
# 5. DROP "ID-LIKE" COLUMNS FROM MODELING FEATURES
#    (keeps them in df, but excludes them from training)

def is_id_like(colname: str) -> bool:
    c = colname.lower()
    return (
        c.endswith("_id") or c in ["id", "session_id", "customer_id", "user_id"] or
        ("uuid" in c) or ("guid" in c)
    )

id_like_cols = [c for c in X_train.columns if is_id_like(c)]
if id_like_cols:
    print("\nID-like columns excluded from modeling:", id_like_cols)

X_train_model = X_train.drop(columns=id_like_cols)
X_test_model  = X_test.drop(columns=id_like_cols)



ID-like columns excluded from modeling: ['session_id', 'customer_id']


### 6. IDENTIFY NUMERIC & CATEGORICAL FEATURES

In [9]:
# 6. IDENTIFY NUMERIC & CATEGORICAL FEATURES (AUTO)

numeric_features = X_train_model.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train_model.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# Sometimes 0/1 flags are numeric; that's fine—they can stay numeric.
print("\nNumeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

# Safety: remove any accidental target leakage columns if they slipped in
for c in LEAKAGE_COLS + dup_target_cols:
    if c in numeric_features: numeric_features.remove(c)
    if c in categorical_features: categorical_features.remove(c)


Numeric features: 6
Categorical features: 6


### 7. SELECT SKEWED NUMERIC FEATURES (TRAIN ONLY) FOR LOG TRANSFORM

In [10]:
# 7. SELECT SKEWED NUMERIC FEATURES (TRAIN ONLY) FOR LOG TRANSFORM
#   Log-transform only numeric columns that are skewed.

# Compute skew on training data only (avoids peeking at test distribution)
skewness = X_train_model[numeric_features].skew(numeric_only=True).sort_values(ascending=False)
SKEW_THRESHOLD = 1.0

skewed_numeric = skewness[skewness.abs() > SKEW_THRESHOLD].index.tolist()
not_skewed_numeric = [c for c in numeric_features if c not in skewed_numeric]

print("\nSkewed numeric features (|skew| > 1.0):", len(skewed_numeric))
print(skewed_numeric)


Skewed numeric features (|skew| > 1.0): 0
[]


### 8. PREPROCESSING PIPELINE (LEAKAGE-FREE)

In [11]:
# 8. PREPROCESSING PIPELINE (LEAKAGE-FREE)
#    - Impute missing values
#    - Log transform skewed numeric
#    - Scale numeric
#    - One-hot encode categoricals


def signed_log1p(X):
    """
    Safe log transform that handles zeros and negatives:
    sign(x) * log1p(abs(x))
    """
    X = np.asarray(X, dtype=float)
    return np.sign(X) * np.log1p(np.abs(X))

log_transformer = FunctionTransformer(signed_log1p, feature_names_out="one-to-one")

# Pipeline for skewed numeric
skewed_num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log", log_transformer),
    ("scaler", StandardScaler())
])

# Pipeline for non-skewed numeric
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline for categorical
cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("skew_num", skewed_num_pipeline, skewed_numeric),
        ("num",      num_pipeline,       not_skewed_numeric),
        ("cat",      cat_pipeline,       categorical_features)
    ],
    remainder="drop"
)

### 9. DEFINE MODELS + GRIDS

In [12]:
# 9) DEFINE MODELS + GRIDS

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LogisticRegression": {
        "estimator": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
        "params": {
            "model__C": [0.1, 1.0, 10.0],
            "model__solver": ["liblinear"]  # stable for small/medium problems
        }
    },
    "RandomForest": {
        "estimator": RandomForestClassifier(random_state=42, class_weight="balanced_subsample"),
        "params": {
            "model__n_estimators": [300, 600],
            "model__max_depth": [None, 8, 16],
            "model__min_samples_leaf": [1, 5, 10]
        }
    },
    "GradientBoosting": {
        "estimator": GradientBoostingClassifier(random_state=42),
        "params": {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth": [2, 3]
        }
    },
    "KNN": {
        "estimator": KNeighborsClassifier(),
        "params": {
            "model__n_neighbors": [5, 15, 35],
            "model__weights": ["uniform", "distance"],
            "model__metric": ["minkowski"]
        }
    }
}

SCORING = "roc_auc"   # primary; also report PR-AUC and others on test

### 10. TRAIN + TUNE (GRIDSEARCHCV) + EVALUATE ON TEST

In [13]:
# 10. TRAIN + TUNE (GRIDSEARCHCV) + EVALUATE ON TEST

results = []
best_estimators = {}

for name, cfg in models.items():
    print("\n" + "="*70)
    print(f"Training: {name}")
    print("="*70)

    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", cfg["estimator"])
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=cfg["params"],
        scoring=SCORING,
        cv=cv,
        n_jobs=-1,
        verbose=0,
        refit=True
    )

    grid.fit(X_train_model, y_train)

    best_pipe = grid.best_estimator_
    best_estimators[name] = best_pipe

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best params:", grid.best_params_)

    # Predict probabilities for evaluation metrics
    y_proba = best_pipe.predict_proba(X_test_model)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    roc = roc_auc_score(y_test, y_proba)
    pr  = average_precision_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    results.append({
        "model": name,
        "best_cv_roc_auc": grid.best_score_,
        "test_roc_auc": roc,
        "test_pr_auc": pr,
        "test_precision@0.5": prec,
        "test_recall@0.5": rec,
        "test_f1@0.5": f1
    })

    print("\nTest Metrics @ threshold=0.5")
    print(f"ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}")
    print(f"Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))


Training: LogisticRegression


/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Best CV ROC-AUC: 1.0
Best params: {'model__C': 0.1, 'model__solver': 'liblinear'}

Test Metrics @ threshold=0.5
ROC-AUC: 1.0000 | PR-AUC: 1.0000
Precision: 0.9906 | Recall: 1.0000 | F1: 0.9953

Confusion Matrix:
[[21525    80]
 [    0  8395]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9963    0.9981     21605
           1     0.9906    1.0000    0.9953      8395

    accuracy                         0.9973     30000
   macro avg     0.9953    0.9981    0.9967     30000
weighted avg     0.9974    0.9973    0.9973     30000


Training: RandomForest


/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_enc

Best CV ROC-AUC: 0.997395350928624
Best params: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 600}


...

Best CV ROC-AUC: 0.9999997917133493
Best params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}

Test Metrics @ threshold=0.5
ROC-AUC: 1.0000 | PR-AUC: 1.0000
Precision: 0.9990 | Recall: 1.0000 | F1: 0.9995

Confusion Matrix:
[[21597     8]
 [    0  8395]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9996    0.9998     21605
           1     0.9990    1.0000    0.9995      8395

    accuracy                         0.9997     30000
   macro avg     0.9995    0.9998    0.9997     30000
weighted avg     0.9997    0.9997    0.9997     30000


Training: KNN


/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Best CV ROC-AUC: 0.9518906906756239
Best params: {'model__metric': 'minkowski', 'model__n_neighbors': 35, 'model__weights': 'distance'}


/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(



Test Metrics @ threshold=0.5
ROC-AUC: 0.9540 | PR-AUC: 0.8744
Precision: 0.8818 | Recall: 0.5839 | F1: 0.7026

Confusion Matrix:
[[20948   657]
 [ 3493  4902]]

Classification Report:
              precision    recall  f1-score   support

           0     0.8571    0.9696    0.9099     21605
           1     0.8818    0.5839    0.7026      8395

    accuracy                         0.8617     30000
   macro avg     0.8694    0.7768    0.8062     30000
weighted avg     0.8640    0.8617    0.8519     30000



### 11. MODEL COMPARISON TABLE

In [15]:
# 11) MODEL COMPARISON TABLE

results_df = pd.DataFrame(results).sort_values(by="test_roc_auc", ascending=False)
print("\n" + "="*70)
print("MODEL COMPARISON (sorted by test ROC-AUC)")
print("="*70)
display(results_df)


MODEL COMPARISON (sorted by test ROC-AUC)


,model,best_cv_roc_auc,test_roc_auc,test_pr_auc,test_precision@0.5,test_recall@0.5,test_f1@0.5
0,LogisticRegression,1.000000,1.000000,1.000000,0.990560,1.000000,0.995258
2,GradientBoosting,1.000000,1.000000,1.000000,0.999048,1.000000,0.999524
1,RandomForest,0.997395,0.997877,0.994186,0.942334,0.981060,0.961307
3,KNN,0.951891,0.953984,0.874367,0.881813,0.583919,0.702594
